In [ ]:
%%capture
# Install dependencies
%pip install pandas numpy matplotlib seaborn python-calamine scikit-learn

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
from seaborn import heatmap
from sklearn import metrics
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

### Import and inspect Dataset

- Read our dataset from the Excel file
- Check if any cells contain invalid data
- List all column titles
- Read excerpt of the data

In [ ]:
X = pd.read_excel('dataset-uci.xlsx', engine='calamine')

print("Missing values:", X.isnull().sum().sum())
print("\nColumn names:\n", X.columns.tolist())

# Stop Pandas from dropping columns in the visualization
pd.set_option('display.max_columns', None)
X.describe()

### Check Data for Outliers

As some data had out of the norm max or minimum values we decided to drop these rows completely

In [ ]:
X = X.drop(318) # extreme High Density Lipoprotein (HDL)
X = X.drop(239) # extreme Obesity value 1954%
X = X.drop(205) # extreme low muscle mass 4,7kg

Hide columns with non-discrete data and create histograms of the different columns to visualize the data and understand its meaning

In [ ]:
items = X.columns.tolist()
to_remove = {'Gallstone Status', 'Gender', 'Comorbidity', 'Coronary Artery Disease (CAD)', 'Hypothyroidism', 'Hyperlipidemia', 'Diabetes Mellitus (DM)'}

items = [x for x in items if x not in to_remove]

X[items].hist(figsize=(20, 18), bins=20)

plt.tight_layout()
plt.show()

### Clean up Columns
- Extract the "Gallstone Status" column as result reference
- Drop columns mathematically related to others, and thus not providing additional information.

In [ ]:
# Extract results
y = X["Gallstone Status"]
X_reduced = X.drop(columns=['Gallstone Status'])

# Same as Body Mass Index (BMI)
X_reduced = X_reduced.drop(columns=['Height', 'Weight', 'Obesity (%)'])

# Same as Total Body Water (TBW)
X_reduced = X_reduced.drop(columns=['Extracellular Water (ECW)', 'Intracellular Water (ICW)', 'Extracellular Fluid/Total Body Water (ECF/TBW)'])

# Same as Total Body Fat Ratio (TBFR) (%)
X_reduced = X_reduced.drop(columns=['Total Fat Content (TFC)', 'Lean Mass (LM) (%)'])

# Same as Visceral Fat Area (VFA)
X_reduced = X_reduced.drop(columns=['Visceral Fat Rating (VFR)'])

print(f"Features reduced from {X.shape[1]} to {X_reduced.shape[1]}")

**Before and After Column Cleanup**

Visualize the dependence of multiple columns on each other before and after dropping the mathematically related ones

In [ ]:
plt.figure(figsize=(12, 10))
heatmap(X.corr(), cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix Original Dataset")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 10))
heatmap(X_reduced.corr(), cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix Reduced Dataset")
plt.tight_layout()
plt.show()

### Split Dataset into Training and Testing Data

In [ ]:
random_state = 7
X_train, X_test, y_train, y_test = train_test_split( X_reduced, y, test_size=0.3, random_state=random_state, stratify=y)
print ('Train set:', X_train.shape,  y_train.shape)
print ('Test set:', X_test.shape,  y_test.shape)

**Class Balance**

Check how the two classes are distributed overall and in the training and test sets

In [ ]:
print("Gallstone Status counts (all data):\n", y.value_counts())
print("\nTrain:\n", y_train.value_counts())
print("\nTest:\n", y_test.value_counts())

### Create Models

**Decision Tree**

Create model, train on training data, execute on test data

In [ ]:
decisionTree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=random_state)
decisionTree.fit(X_train,y_train)

Calculate Accuracy and Precision

In [ ]:
predDecisonTree = decisionTree.predict(X_test)
print("Decision Tree's Accuracy: ", metrics.accuracy_score(y_test, predDecisonTree))
print("Decision Tree's Precision: ", metrics.precision_score(y_test, predDecisonTree))

**Random Forest**

Create model, train on training data, execute on test data

In [ ]:
randomForest = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=3, random_state=random_state)
randomForest.fit(X_train,y_train)

Calculate Accuracy and Precision

In [ ]:
predRandomForest = randomForest.predict(X_test)

print("Random Forest's Accuracy: ", metrics.accuracy_score(y_test, predRandomForest))
print("Random Forest's Precision: ", metrics.precision_score(y_test, predRandomForest))

**Gradient Boosting**

Create model, train on training data, execute on test data

In [ ]:
gradientBoosting = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=random_state)
gradientBoosting.fit(X_train,y_train)

Calculate Accuracy and Precision

In [ ]:
predGradientBoosting = gradientBoosting.predict(X_test)

print("Gradient Boosting's Accuracy: ", metrics.accuracy_score(y_test, predGradientBoosting))
print("Gradient Boosting's Precision: ", metrics.precision_score(y_test, predGradientBoosting))

### Additional Evaluation

Besides accuracy and precision, look at recall and F1 per class and at the confusion matrices

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

predictions = {
    'Decision Tree': predDecisonTree,
    'Random Forest': predRandomForest,
    'Gradient Boosting': predGradientBoosting,
}

for name, pred in predictions.items():
    print(f"{name}:\n", classification_report(y_test, pred), "\n")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

**Cross-Validation**

A single split on a small test set is not very stable, so we also run a 5-fold stratified cross-validation on the whole reduced dataset

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

models = {
    'Decision Tree': decisionTree,
    'Random Forest': randomForest,
    'Gradient Boosting': gradientBoosting,
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
for name, model in models.items():
    scores = cross_val_score(model, X_reduced, y, cv=cv, scoring='accuracy')
    print(f"{name}: accuracy {scores.mean():.4f} (+/- {scores.std():.4f})")

**Permutation Importance**

Check if the top features stay the same when using permutation importance on the test data instead of the built-in importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, (name, model) in zip(axes, models.items()):
    result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=random_state)
    perm_imp = pd.Series(result.importances_mean, index=X_reduced.columns).sort_values() # type: ignore
    perm_imp.tail(5).plot(kind='barh', ax=ax)
    ax.set_title(f"{name} Permutation Importance")
    ax.set_xlabel("Mean accuracy decrease")

plt.tight_layout()
plt.show()

### Visualizations

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(decisionTree, feature_names=list(X_reduced.columns), class_names=['No Gallstone', 'Gallstone'], filled=True, rounded=True, impurity=False, fontsize=10)
plt.tight_layout()
plt.show()